## Building a Chatbot
In this section we'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.
Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for.
* Conversational RAG: Enable a chatbot experience over an external source of data.
* Agents: Build a chatbot that can take actions

This section tutorial will cover the basics which will be helpful for those two more advanced topics.


In [2]:
import os
from dotenv import load_dotenv
load_dotenv() # loading all the environment variables
groq_api_key=os.getenv("GROQ_API_KEY")

In [4]:
from langchain_groq import ChatGroq
model=ChatGroq(model='llama-3.3-70b-versatile',api_key=groq_api_key)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x106afdbe0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x106afe900>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [6]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi, My name is Gaurav and I am a software Engineer!")])

AIMessage(content="Nice to meet you, Gaurav! Welcome! It's great to hear that you're a software engineer. What kind of projects do you usually work on, or what technologies are you most interested in? I'm here to chat and help with any questions or topics you'd like to discuss!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 50, 'total_tokens': 111, 'completion_time': 0.176633397, 'completion_tokens_details': None, 'prompt_time': 0.019537956, 'prompt_tokens_details': None, 'queue_time': 0.161963303, 'total_time': 0.196171353}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fb3c9-7286-7da1-acbd-1437bf104e80-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 50, 'output_tokens': 61, 'total_tokens': 111})

In [7]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi, My name is Gaurav and I am a software Engineer!"),
        AIMessage(content="Nice to meet you, Gaurav! Welcome! It's great to hear that you're a software engineer. What kind of projects do you usually work on, or what technologies are you most interested in? I'm here to chat and help with any questions or topics you'd like to discuss!"),
        HumanMessage(content="Hey, what's my name and what my profession"),
    ]
)

AIMessage(content="Your name is Gaurav, and you're a Software Engineer.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 130, 'total_tokens': 145, 'completion_time': 0.039407319, 'completion_tokens_details': None, 'prompt_time': 0.015781591, 'prompt_tokens_details': None, 'queue_time': 0.059004029, 'total_time': 0.05518891}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fb3ca-6136-7ab3-854d-f665d28612ff-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 130, 'output_tokens': 15, 'total_tokens': 145})

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this.

In [14]:
from  langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import  BaseChatMessageHistory
from langchain_core.runnables import RunnableWithMessageHistory

store={}
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]
with_message_history=RunnableWithMessageHistory(
    model,
    get_session_history
)


In [22]:
config={
    "configurable":{
        "session_id":"chat1"
    }
}

In [17]:
response=with_message_history.invoke(
    [
        HumanMessage(content="Hi, My name is Gaurav and I am a software Engineer!"),

     ],
    config=config
)

In [18]:
response.content

"Hello again Gaurav, nice to meet you once more. It seems like we already started a conversation earlier. Would you like to continue where we left off or start fresh? What's on your mind, and how can I assist you today as a software engineer?"

In [23]:
response=with_message_history.invoke(
    [
        HumanMessage(content="Hi, What is my name?"),

     ],
    config=config
)
response.content


'Your name is Gaurav.'